In [1]:
import trimesh
import numpy as np
import os
import pandas as pd

In [2]:
df = pd.read_excel(r"D:\python\brain_areas.xlsx")

In [3]:
df.loc[df.region == "ACB"]

,ID,region,full,level,group
575,56,ACB,Nucleus accumbens,7,STR


In [4]:


def crop_left_brain_mesh(input_path, output_path, axis='z', keep='min', custom_midline=None):
    """
    精确截取脑区左侧部分 (默认针对 Z 轴即左右轴)
    
    :param axis: 左右方向对应的轴，默认为 'z'
    :param keep: 'min' 保留坐标值小的一侧，'max' 保留坐标值大的一侧
    :param custom_midline: 若指定数值（如 5700），则以此绝对坐标为脑中线；若为 None 则按当前模型中心
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # 1. 加载并合并网格
    loaded = trimesh.load(input_path)
    if isinstance(loaded, trimesh.Scene):
        mesh = trimesh.util.concatenate(loaded.dump())
    else:
        mesh = loaded

    if mesh.is_empty:
        print(f"错误: 无法读取模型 -> {input_path}")
        return

    axis_idx = {'x': 0, 'y': 1, 'z': 2}[axis.lower()]
    min_b, max_b = mesh.bounds[0], mesh.bounds[1]
    
    print("=" * 50)
    print(f"【模型坐标范围】")
    print(f"  X 轴 (前后): {min_b[0]:.2f} ~ {max_b[0]:.2f}")
    print(f"  Y 轴 (上下): {min_b[1]:.2f} ~ {max_b[1]:.2f}")
    print(f"  Z 轴 (左右): {min_b[2]:.2f} ~ {max_b[2]:.2f}")

    # 2. 确定中线分割坐标
    if custom_midline is not None:
        split_val = custom_midline
    else:
        # 使用当前脑区在左右轴上的几何中心
        split_val = (min_b[axis_idx] + max_b[axis_idx]) / 2.0

    print(f"【分割设置】沿 {axis.upper()} 轴切割，中线位置 = {split_val:.2f}，保留 {keep.upper()} 侧")

    # 3. 采用面过滤（Face Masking），100% 物理移除另一侧的面，绝不会失效
    # 获取每个三角面中心的坐标
    face_centers = mesh.triangles_center[:, axis_idx]
    
    if keep == 'min':
        keep_mask = face_centers <= split_val
    else:
        keep_mask = face_centers >= split_val

    # 提取过滤后的子网格
    submesh = mesh.submesh([keep_mask], append=True)

    # 4. 导出保存
    submesh.export(output_path)
    print(f"【处理成功】原始面数: {len(mesh.faces)} -> 剩余面数: {len(submesh.faces)}")
    print(f"结果已保存至: {output_path}")
    print("=" * 50)


if __name__ == "__main__":
    input_file = r"D:\python\neuron-vis\resource\allobj\56.obj"
    output_file = r"J:\BLA_three_types\terminal_new_name\ACB.obj"

    crop_left_brain_mesh(
        input_path=input_file,
        output_path=output_file,
        axis='z',        # 鼠脑坐标左右方向为 Z 轴
        keep='min',      # 若切出来发现保留的是右侧，将此项改为 'max'
        custom_midline=None # 如果是标准全脑空间(10um)，鼠脑正中线通常为 5700，可设为 5700
    )

【模型坐标范围】
  X 轴 (前后): 3457.10 ~ 5042.13
  Y 轴 (上下): 4661.59 ~ 6616.11
  Z 轴 (左右): 3185.29 ~ 8177.40
【分割设置】沿 Z 轴切割，中线位置 = 5681.34，保留 MIN 侧
【处理成功】原始面数: 5280 -> 剩余面数: 2636
结果已保存至: J:\BLA_three_types\terminal_new_name\ACB.obj
